In [84]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import os
from PIL import Image

In [85]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [86]:
class CarsVsBikes(Dataset):
    def __init__(self, path_dir1, path_dir2, transform=None):
        self.transform = transform
        self.car_paths = [(os.path.join(path_dir1, f)) for f in sorted(os.listdir(path_dir1))]
        self.bike_paths = [(os.path.join(path_dir2, f)) for f in sorted(os.listdir(path_dir2))]

    def __len__(self):
        return len(self.car_paths) + len(self.bike_paths)

    def __getitem__(self, i):
        if i < len(self.car_paths):
            label = 0
            img_path = self.car_paths[i]
        else:
            label = 1
            img_path = self.bike_paths[i - len(self.car_paths)]

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        
        label = torch.tensor(label)

        return img, label

In [87]:
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
            mean=[0.5,0.5,0.5],
            std=[0.5,0.5,0.5]
        ),
])

transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [88]:
cars_path_train = "./images/Car/train"
bikes_path_train = "./images/Bike/train"

cars_path_test = "./images/Car/test"
bikes_path_test = "./images/Bike/test"

cars_vs_bikes_train_dataset = CarsVsBikes(cars_path_train, bikes_path_train, transform_train)
cars_vs_bikes_test_dataset = CarsVsBikes(cars_path_test, bikes_path_test, transform_test)

In [89]:
train_loader = DataLoader(dataset=cars_vs_bikes_train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(dataset=cars_vs_bikes_test_dataset, batch_size=8, shuffle=True) 

In [90]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # 256x256
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),


            # 128x128
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),


            # 64x64
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),


            # 32x32
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),

            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128,2)
        )


    def forward(self,x):
        x=self.features(x)
        x=self.classifier(x)
        return x

In [91]:
model = CNN()
model = model.to(device)

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [92]:
epochs = 20

for epoch in range(epochs):
    for inputs, labels in train_loader:
        #print(labels.unique(return_counts=True))
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = loss_func(outputs, labels)
        loss.backward()
        #print(model.feautures[0].weight.grad.abs().mean())
        optimizer.step()

        optimizer.zero_grad()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch: {epoch + 1}/{epochs}\nLoss: {loss.item():.4f}")  

Epoch: 5/20
Loss: 0.2821
Epoch: 10/20
Loss: 0.2210
Epoch: 15/20
Loss: 0.1497
Epoch: 20/20
Loss: 0.0104


In [93]:

correct, total = 0, 0

model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        prediction = model(inputs)
        _, predicted = torch.max(prediction, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy*100:.2f}%")

Test Accuracy: 96.62%


In [ ]:

classes = {
    0:"Car",
    1:"Bike",
}

model.eval()

image = Image.open("./bike.jpg")
transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Resize((256, 256))])
image_tensor = transform(image)


with torch.no_grad():
    predicted_class = model(image_tensor.unsqueeze(0).to(device))
    predicted = torch.argmax(predicted_class)
classes.get(int(predicted))

'Bike'

In [96]:

torch.save(model.state_dict, 'model.pt')